# Lab 6 — Decision Trees and Ensembles
**Coverage:** Chapters 12–13

This notebook is one of the ten course labs. Complete the core activities in order; transfer activities are optional extensions inside the same lab and do not create additional lab numbers.


## Part A — Interpretable decision tree on Titanic
**Core activity.**


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "all_datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
train_path = PROJECT_ROOT / "all_datasets" / "titanic_dataset" / "Titanic-Dataset.csv"
df = pd.read_csv(train_path)

In [ ]:
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = df[features]
y = df["Survived"]
num_cols = ["Age", "SibSp", "Parch", "Fare"]
cat_cols = ["Pclass", "Sex", "Embarked"]

preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
model = Pipeline([
    ("preprocess", preprocess),
    ("tree", DecisionTreeClassifier(
        max_depth=4, min_samples_leaf=12, random_state=42
    )),
])

In [ ]:
model.fit(X_train, y_train)
train_pred = model.predict(X_train)
valid_pred = model.predict(X_valid)
print("training accuracy:", round(accuracy_score(y_train, train_pred), 3))
print("validation accuracy:", round(accuracy_score(y_valid, valid_pred), 3))

In [ ]:
feature_names = model.named_steps["preprocess"].get_feature_names_out()
print(export_text(
    model.named_steps["tree"],
    feature_names=list(feature_names),
    max_depth=3,
))

In [ ]:
# Plot only the top levels so the learned rules remain readable.
feature_names = model.named_steps["preprocess"].get_feature_names_out()
plt.figure(figsize=(12, 6))
plot_tree(
    model.named_steps["tree"],
    feature_names=feature_names,
    class_names=["did not survive", "survived"],
    max_depth=2,
    filled=True,
    fontsize=8,
)
plt.title("Top of the fitted Titanic decision tree")
plt.tight_layout()
plt.show()

## Part B — Ames Housing ensemble regression
**Core activity.**


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "all_datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
train_path = PROJECT_ROOT / "all_datasets" / "ames_housing_dataset" / "AmesHousing.csv"
df = pd.read_csv(train_path)

In [ ]:
# Order and PID are identifiers, not useful house characteristics for this lesson.
X = df.drop(columns=["SalePrice", "Order", "PID"], errors="ignore")
y = np.log1p(df["SalePrice"])

num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(exclude="number").columns
preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), cat_cols),
])

models = {
    "random_forest": RandomForestRegressor(
        n_estimators=150, max_features=0.8, min_samples_leaf=2,
        random_state=42, n_jobs=-1
    ),
    "gradient_boosting": GradientBoostingRegressor(
        n_estimators=150, learning_rate=0.05, max_depth=3,
        loss="huber", random_state=42
    ),
}
cv = KFold(n_splits=3, shuffle=True, random_state=42)

In [ ]:
for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
    neg_mse = cross_val_score(
        pipe, X, y, cv=cv, scoring="neg_mean_squared_error", n_jobs=1
    )
    rmse = np.sqrt(-neg_mse)
    print(name, "CV log-RMSE:", round(rmse.mean(), 4),
          "+/-", round(rmse.std(), 4))

## Part C — Transfer activity: wine-quality ensembles
**Optional transfer.**


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "all_datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
train_path = PROJECT_ROOT / "all_datasets" / "red_wine_quality_dataset" / "winequality-red.csv"
df = pd.read_csv(train_path)

In [ ]:
X = df.drop(columns="quality")
y = df["quality"]
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=42
)

models = {
    "random_forest": RandomForestRegressor(
        n_estimators=400, min_samples_leaf=2, random_state=42, n_jobs=-1
    ),
    "gradient_boosting": GradientBoostingRegressor(random_state=42),
}

In [ ]:
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_valid)
    results[name] = pred
    print(name)
    print(" MAE:", round(mean_absolute_error(y_valid, pred), 3))
    print(" RMSE:", round(np.sqrt(mean_squared_error(y_valid, pred)), 3))

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(y_valid, results["gradient_boosting"], alpha=0.65)
plt.xlabel("Observed quality")
plt.ylabel("Predicted quality")
plt.title("Wine quality validation: gradient boosting")
plt.tight_layout()
plt.show()